# Telco Customer Churn: Exploratory Data Analysis

Data exploration for the Telco Customer Churn dataset (IBM sample). This notebook downloads the data, inspects its structure, and prepares it for modeling.


## Setup and data download

The dataset is pulled directly from IBM's GitHub repository. No manual download needed.

In [22]:
import os
import urllib.request
import pandas as pd

In [23]:
def load_or_download_data(file_path, url):
    """Load a CSV from `file_path`, downloading it from `url` first if missing.

    Parameters
    ----------
    file_path : str
        Local path where the CSV is (or will be) stored.
    url : str
        Direct URL to download the CSV from if it does not exist locally.

    Returns
    -------
    pandas.DataFrame
        The loaded dataset.
    """
    if os.path.exists(file_path):
        print(f"Loading data from {file_path}")
    else:
        directory = os.path.dirname(file_path)
        if directory and not os.path.exists(directory):
            os.makedirs(directory)
        print(f"Downloading data from {url}")
        urllib.request.urlretrieve(url, file_path)
        print(f"Download complete. File saved to '{file_path}'.")
    return pd.read_csv(file_path)

In [24]:
LOCAL_CHURN_PATH = "../data/raw/customer_churn.csv"
CHURN_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df_churn = load_or_download_data(LOCAL_CHURN_PATH, CHURN_URL)
df_churn.head()

Loading data from ../data/raw/customer_churn.csv


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Basic structure

The dataset has 7,043 rows and 21 columns. Features are a mix of customer demographics, account information, and service subscriptions.

In [25]:
print("Shape:", df_churn.shape)
print(df_churn.dtypes.value_counts())

Shape: (7043, 21)
object     18
int64       2
float64     1
Name: count, dtype: int64


## Churn rate (target distribution)

The churn rate is roughly 26.5%, making this a moderately imbalanced problem rather than the balanced 50/50 split sometimes assumed. This matters for metric selection: plain accuracy is misleading at this ratio, so we will use ROC-AUC and F1 instead.

In [26]:
print(df_churn["Churn"].value_counts())
print()
print(df_churn["Churn"].value_counts(normalize=True).round(3))

Churn
No     5174
Yes    1869
Name: count, dtype: int64

Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64


## Feature types

Most features are stored as text (object dtype). `SeniorCitizen` is stored as an integer but it is really a binary flag (0/1). `TotalCharges` is listed as text even though it should be numeric. That is a red flag worth investigating.

In [27]:
print(df_churn.dtypes)

customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object


## Numerical features

The three numeric columns look reasonable. Tenure caps at 72 months (6 years), which makes sense. MonthlyCharges maxes out at $118.75. No obvious outliers.

Pandas `isna()` reports zero missing values across all columns, but that only catches NaN and None. A column stored as text could hide dirty values that look like valid strings.

In [28]:
print(df_churn.describe())
print()
print(df_churn.isna().sum())

       SeniorCitizen       tenure  MonthlyCharges
count    7043.000000  7043.000000     7043.000000
mean        0.162147    32.371149       64.761692
std         0.368612    24.559481       30.090047
min         0.000000     0.000000       18.250000
25%         0.000000     9.000000       35.500000
50%         0.000000    29.000000       70.350000
75%         0.000000    55.000000       89.850000
max         1.000000    72.000000      118.750000

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


## The hidden missing values

When we force-convert `TotalCharges` from text to numbers, 11 values become NaN. The culprit is a single space character (" ") stored as the string value instead of a proper blank. This is invisible to `isna()` because a space is technically a valid string.

In [29]:
df_churn["TotalCharges"] = pd.to_numeric(df_churn["TotalCharges"], errors="coerce")
print("NaNs after coercion:", df_churn["TotalCharges"].isna().sum())
print()
print("Blank strings found:", (df_churn["TotalCharges"].isna()).sum())

NaNs after coercion: 11

Blank strings found: 11


## Inspecting the 11 dirty rows

Every one of these rows has `tenure = 0`. These are brand new customers who signed up but have not received their first bill yet. The blank `TotalCharges` is not a data entry mistake, it is simply an unset field.

We drop these 11 rows because they represent 0.16% of the data, small enough to have no statistical impact. An equally valid choice would be `fillna(0)` since these customers genuinely owe nothing at this point.

In [30]:
df_churn[df_churn["TotalCharges"].isna()]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,NaN,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,NaN,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,NaN,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,NaN,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,NaN,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,NaN,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,NaN,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,...,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,NaN,No


In [ ]:
df_churn = df_churn.dropna(subset=["TotalCharges"])
os.makedirs("../data/processed", exist_ok=True)
df_churn.to_csv("../data/processed/churn_clean.csv", index=False)
print("Remaining rows:", len(df_churn))
print("TotalCharges NaNs remaining:", df_churn["TotalCharges"].isna().sum())
df_churn.head() 

Remaining rows: 7032
TotalCharges NaNs remaining: 0


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
